In [1]:
from datasets import load_dataset
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("AI-MO/NuminaMath-1.5")

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'problem_is_valid', 'solution_is_valid', 'source', 'synthetic'],
        num_rows: 896215
    })
})

In [3]:
train_ds = ds["train"]

In [4]:
df = train_ds.to_pandas()

In [5]:
df["problem_is_valid"]

0         Yes
1         Yes
2         Yes
3         Yes
4         Yes
         ... 
896210    Yes
896211    Yes
896212    Yes
896213    Yes
896214    Yes
Name: problem_is_valid, Length: 896215, dtype: object

In [6]:
filtered = train_ds.filter(
    lambda x: (
        x["problem_is_valid"] == "Yes"
        and x["solution_is_valid"] == "Yes"
        and x["answer"] is not None
        and x["answer"].strip().lower() != "proof"
    )
)

subset = filtered.shuffle(seed=42).select(range(5000))

In [7]:
df_sub = subset.to_pandas()

In [8]:
df_sub = df_sub["problem"]

In [9]:
df_sub

0       Given that A, B, and C are three distinct poin...
1       Matt orders some pounds of beef. He cuts that ...
2       Simplify the expression $3C_{10}^1 + 9C_{10}^2...
3       Four. (20 points) The sequence $\left\{a_{n}\r...
4       In the Cartesian coordinate system, the point ...
                              ...                        
4995    Given a geometric sequence \\(\{a_n\}\) satisf...
4996    The gravitational attraction that Earth exerts...
4997    Let $f(x) = \cos(\omega x + \varphi)$, where $...
4998    There were 61 parents in the program and some ...
4999    A post office sells stamps of three denominati...
Name: problem, Length: 5000, dtype: object

In [10]:
df_sub.iloc[0]

'Given that A, B, and C are three distinct points on line l, and point O is not on line l, the set of real numbers x that satisfy the equation $x^{2} \\overrightarrow{OA} + x \\overrightarrow{OB} + \\overrightarrow{BC} = \\overrightarrow{0}$ is (\u3000\u3000)\n\nA: $\\{-1\\}$  \nB: $\\emptyset$  \nC: $\\{0\\}$  \nD: $\\{0, -1\\}$'

In [11]:
import re
import pandas as pd

# Regex for common LaTeX math formats
MATH_PATTERN = r"\$\$.*?\$\$|\$.*?\$|\\\\\(.*?\\\\\)|\\\\\[.*?\\\\\]"

def extract_math(text):
    equations = []

    def replacer(match):
        idx = len(equations)
        equations.append(match.group(0))
        return f"<MATH_{idx}>"

    processed_text = re.sub(
        MATH_PATTERN,
        replacer,
        text,
        flags=re.DOTALL
    )

    return pd.Series([processed_text, equations])


# df_sub -> pandas Series containing problem statements
result_df = df_sub.apply(extract_math)

# Rename columns
result_df.columns = [
    "problem_with_placeholders",
    "equations"
]

In [12]:
result_df

,problem_with_placeholders,equations
0,"Given that A, B, and C are three distinct poin...",[$x^{2} \overrightarrow{OA} + x \overrightarro...
1,Matt orders some pounds of beef. He cuts that ...,[]
2,Simplify the expression <MATH_0> (express the ...,[$3C_{10}^1 + 9C_{10}^2 + \ldots + 3^{10}C_{10...
3,Four. (20 points) The sequence <MATH_0> is def...,"[$\left\{a_{n}\right\}$, $a_{1}=3, a_{n}=$, $3..."
4,"In the Cartesian coordinate system, the point ...","[$P(-2,3)$, $x$]"
...,...,...
4995,"Given a geometric sequence <MATH_0>, then <MAT...","[\\(\{a_n\}\) satisfying \\(a_1 = \frac{1}{2},..."
4996,The gravitational attraction that Earth exerts...,[]
4997,"Let <MATH_0>, where <MATH_1>. The smallest pos...","[$f(x) = \cos(\omega x + \varphi)$, $(\omega >..."
4998,There were 61 parents in the program and some ...,[]


In [13]:
result_df.iloc[0]["problem_with_placeholders"]

'Given that A, B, and C are three distinct points on line l, and point O is not on line l, the set of real numbers x that satisfy the equation <MATH_0> is (\u3000\u3000)\n\nA: <MATH_1>  \nB: <MATH_2>  \nC: <MATH_3>  \nD: <MATH_4>'

In [14]:
result_df.iloc[0]["equations"]

['$x^{2} \\overrightarrow{OA} + x \\overrightarrow{OB} + \\overrightarrow{BC} = \\overrightarrow{0}$',
 '$\\{-1\\}$',
 '$\\emptyset$',
 '$\\{0\\}$',
 '$\\{0, -1\\}$']

# Translation ground truth

In [38]:
problem_text = result_df.iloc[2]["problem_with_placeholders"]
problem_text

'Simplify the expression <MATH_0> (express the answer using mathematical notation).'

In [39]:
import os

TOKEN = os.environ["PCSS_API_KEY"]

In [43]:
%%time
import requests
import json
 
# LiteLLM endpoint for chat completions
url = "https://llm.hpc.psnc.pl/v1/chat/completions"
 
headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

messages = [
    {
        "role": "system",
        "content": (
            "You are a professional mathematical translator.\n"
            "Translate English math problems into Polish.\n"
            "\n"
            "Rules:\n"
            "- Preserve all placeholders like <MATH_0>, <MATH_1> exactly unchanged.\n"
            "- Do not modify placeholder positions.\n"
            "- Do not translate or alter mathematical expressions.\n"
            "- Do not solve the problem.\n"
            "- Do not add explanations.\n"
            "- Do not add notes.\n"
            "- Do not add quotation marks.\n"
            "- Do not add markdown.\n"
            "- Output ONLY the translated text.\n"
            "- Preserve line breaks and option formatting.\n"
        )
    },
    {
        "role": "user",
        "content": result_df.iloc[0]["problem_with_placeholders"]
    },
    {
        "role": "assistant",
        "content": '\n\nDane są trzy różne punkty A, B i C na prostej l, a punkt O nie leży na prostej l. Zbiór liczb rzeczywistych x spełniających równanie <MATH_0> to (  )\n\nA: <MATH_1>\nB: <MATH_2>\nC: <MATH_3>\nD: <MATH_4>'
    },
    {
        "role": "user",
        "content": result_df.iloc[1]["problem_with_placeholders"]
    },
    {
        "role": "assistant",
        "content": '\n\nMatt zamawia pewną liczbę funtów wołowiny. Kroi ją na steki po 12 uncji i otrzymuje 20 steków. Ile funtów wołowiny zamówił?'
    },
    {
        "role": "user",
        "content": problem_text
    }
]
 
# Request payload
data = {
    "model": "Qwen3.5-397B-A17B",
    "messages": messages
}

response = requests.post(url, headers=headers, json=data)
 
if response.status_code == 200:
    # Pretty print full JSON
    print(json.dumps(response.json(), indent=4))
     
    # Extract and print just the model's reply
    reply = response.json()["choices"][0]["message"]["content"]
    print("\nModel reply:", reply)
else:
    print(f"Error {response.status_code}: {response.text}")

CPU times: user 39.6 ms, sys: 7.32 ms, total: 46.9 ms
Wall time: 6min 25s


KeyboardInterrupt: 

In [36]:
print(problem_text)

Matt orders some pounds of beef. He cuts that into 12-ounce steaks and gets 20 steaks. How many pounds of beef did he order?


In [33]:
print(reply)



Matt zamawia pewną liczbę funtów wołowiny. Kroi ją na steki po 12 uncji i otrzymuje 20 steków. Ile funtów wołowiny zamówił?


In [34]:
print(problem_text)

Matt orders some pounds of beef. He cuts that into 12-ounce steaks and gets 20 steaks. How many pounds of beef did he order?
